# AutoPhOT example setup

End-to-end AutoPhOT run on a folder of FITS images: configure, run, and plot the resulting light curve. Each numbered section sets one group of parameters. A table lists what every key does, then a code cell applies it. Edit the paths and the target first; keep the other values until you have a reason to change them.

### Before you start

- AutoPhOT installed (`conda install -c conda-forge -c astro-sean autophot`, or `pip install .` from a source checkout)
- FITS images carrying `TELESCOP`, `INSTRUME`, and a bandpass keyword such as `FILTER`
- SExtractor on `PATH` when running template subtraction
- MAST CasJobs credentials only if you plan to use `refcat`

The complete parameter reference lives in `databases/default_input.yml`; `list_parameters()` prints every key with its default.

## How this notebook is organized

`AutomatedPhotometry.load()` returns every config key as one nested dict, `autophot_input`. Each section overrides one group:

1. Paths and target
2. Calibration catalog
3. Preprocessing and photometry
4. WCS and distortion
5. Limiting magnitudes
6. Template subtraction
7. Parallelism and verbosity
8. Run
9. Light curve and tables

Run the cells top to bottom.

## Setup

Imports and the default config. `_HAS_GPU` is detected once here; later sections use it for the SFFT backend and MaxiMask batch sizing.

In [ ]:
import glob
import os
import sys

# Only needed when running from a source checkout instead of an installed package.
# sys.path.append('/path/to/autophot')

from autophot import AutomatedPhotometry

autophot_input = AutomatedPhotometry.load()

_HAS_GPU = os.path.exists('/dev/kfd') or bool(glob.glob('/dev/nvidia*'))

## 1. Paths and target

`fits_dir` is scanned recursively for FITS images; flat and nested layouts both work (see Folder layout in the README). Outputs go to a `<fits_dir>_<outdir_name>` sibling.

| Key | Set to | Purpose |
|---|---|---|
| `fits_dir` | `'/path/to/photometry'` | Input directory holding the FITS images |
| `wdir` | `'/path/to/autophot_db'` | Working directory for caches, catalog queries, and the telescope database |
| `outdir_name` | `'REDUCED'` | Suffix of the mirrored output directory |
| `target_name` | `'SN2026abc'` | Resolved through TNS; set `target_ra` / `target_dec` to skip the lookup |
| `selected_filters` | `['g', 'r', 'i']` | Bands to process; an empty list processes everything |
| `restart` | `False` | `False` skips images that already produced an `Output_*.csv` |

Optional keys shown commented in the code:

- `target_ra` / `target_dec`: explicit coordinates instead of a TNS lookup
- `additional_targets` / `additional_targets_ra_dec`: extra sources fitted alongside the primary target

In [ ]:
autophot_input['fits_dir'] = '/path/to/photometry'
autophot_input['wdir'] = '/path/to/autophot_db'
autophot_input['outdir_name'] = 'REDUCED'

autophot_input['target_name'] = 'SN2026abc'
# autophot_input['target_ra'] = 123.456789
# autophot_input['target_dec'] = -12.345678
# autophot_input['additional_targets'] = ['SN2021xyz']
# autophot_input['additional_targets_ra_dec'] = [[123.4, -12.3]]

autophot_input['selected_filters'] = ['g', 'r', 'i']
autophot_input['restart'] = False

## 2. Calibration catalog

The catalog supplies reference magnitudes for the zeropoint fit.

| Key | Set to | Purpose |
|---|---|---|
| `catalog.use_catalog` | `'auto'` | Catalog backend: one name, `'auto'`, or a band-to-catalog dict |
| `catalog.MASTcasjobs_wsid` | `'...'` | MAST workspace ID for `refcat` queries |
| `catalog.MASTcasjobs_pwd` | `'...'` | MAST token for `refcat` queries |
| `catalog.transmission_curve_map` | `{'band': path}` | Local throughput curves for `gaia_custom` bands |
| `catalog.curve_map_svo` | `{'band': id}` | SVO Filter Profile Service IDs to fetch curves from |

Notes on `use_catalog` values:

- `'auto'` measures each image footprint once and keeps the catalog with the most usable calibrators per band; the choice is cached under `<wdir>/catalog_queries/`
- Single backend names include `'gaia'`, `'panstarrs'`, `'sdss'`, `'apass'`, `'refcat'`, `'2mass'`, `'skymapper'`, `'legacy'`
- A dict maps band groups to backends, e.g. `{'grizJHK': 'refcat', 'UBVRI': 'apass', 'default': 'gaia'}`; case matters (`r` is SDSS r, `R` is Johnson-Cousins R)
- `'gaia_custom'` convolves Gaia XP spectra with your throughput curves for bands no survey covers

In [ ]:
autophot_input['catalog']['use_catalog'] = 'auto'
# autophot_input['catalog']['use_catalog'] = {
#     'grizJHK': 'refcat',
#     'u': 'gaia',
#     'UBVRI': 'apass',
#     'default': 'gaia',
# }

# autophot_input['catalog']['MASTcasjobs_wsid'] = '...'
# autophot_input['catalog']['MASTcasjobs_pwd'] = '...'

# autophot_input['catalog']['transmission_curve_map'] = {'myband': '/path/to/myband.dat'}
# autophot_input['catalog']['curve_map_svo'] = {'r': 'SDSS/SDSS.r'}

## 3. Preprocessing and photometry

Image cleaning and the photometric measurement itself.

| Key | Set to | Purpose |
|---|---|---|
| `cosmic_rays.remove_cmrays` | `0` | Cosmic-ray rejection; slow on large frames |
| `preprocessing.trim_image` | `5` | Crop to an arcmin box around the target |
| `background.use_maximask` | `False` | CNN defect mask on top of the heuristic masks; needs `maximask-and-maxitrack` and tensorflow |
| `background.maximask_allow_gpu` | `_HAS_GPU` | Allow MaxiMask on GPU; far faster than CPU |
| `background.maximask_batch_size` | `32` or `4` | Inference batch size, GPU vs CPU |
| `photometry.psf_oversample` | `1` | ePSF oversampling factor |
| `photometry.perform_emcee_fitting_s2n` | `100000` | S/N below which the MCMC fitter takes over; a huge value runs MCMC on every detection |
| `photometry.apply_colorterms` | `False` | Apply catalog color terms in the zeropoint fit |
| `photometry.psf_spatial_grid` | `'auto'` or N | Field-varying PSF on an NxN grid for wide or distorted fields |

In [ ]:
autophot_input['cosmic_rays']['remove_cmrays'] = 0
autophot_input['preprocessing']['trim_image'] = 5

autophot_input['background']['use_maximask'] = False
autophot_input['background']['maximask_allow_gpu'] = _HAS_GPU
autophot_input['background']['maximask_batch_size'] = 32 if _HAS_GPU else 4

autophot_input['photometry']['psf_oversample'] = 1
autophot_input['photometry']['perform_emcee_fitting_s2n'] = 100000
autophot_input['photometry']['apply_colorterms'] = False
# autophot_input['photometry']['psf_spatial_grid'] = 'auto'

## 4. WCS and distortion

Astrometry and optical-distortion handling.

| Key | Set to | Purpose |
|---|---|---|
| `wcs.redo_wcs` | `1` | Re-solve each frame with astrometry.net |
| `wcs.solver` | `'scamp'` | Use SCAMP + SExtractor instead of astrometry.net |
| `wcs.correct_distortion` | `False` | SCAMP + SWarp resampling for fields with real optical distortion |

- `correct_distortion` needs `scamp` and `swarp` on `PATH`; on failure the frame is processed uncorrected
- On undersampled data set `correct_distortion_resampling_type = 'LANCZOS2'`; severe distortion may warrant `scamp_distort_degrees = 4`

In [ ]:
autophot_input['wcs']['redo_wcs'] = 1
# autophot_input['wcs']['solver'] = 'scamp'
autophot_input['wcs']['correct_distortion'] = False

## 5. Limiting magnitudes

Artificial stars are injected and recovered at the configured S/N thresholds.

| Key | Set to | Purpose |
|---|---|---|
| `limiting_magnitude.n_jobs` | `8` | Workers for the injection/recovery bisection |
| `limiting_magnitude.inject_final_n_sites` | `4` | Farthest-point-spaced injection sites per iteration |
| `limiting_magnitude.recovery_method` | `'PSF'` | Photometry method used to recover injected stars |
| `limiting_magnitude.uncal_apply` | `True` | Rescale reported errors by the injection z-score scatter |

In [ ]:
autophot_input['limiting_magnitude']['n_jobs'] = 8
autophot_input['limiting_magnitude']['inject_final_n_sites'] = 4
# autophot_input['limiting_magnitude']['recovery_method'] = 'PSF'
# autophot_input['limiting_magnitude']['uncal_apply'] = True

## 6. Template subtraction

Difference imaging against a reference frame.

| Key | Set to | Purpose |
|---|---|---|
| `template_subtraction.do_subtraction` | `1` | Enable the subtraction stage |
| `template_subtraction.alignment_method` | `'spalipy'` | Template registration method |
| `template_subtraction.method` | `'sfft'` | `'sfft'`, `'hotpants'`, `'zogy'`, or `'auto'` |
| `template_subtraction.sfft_backend` | `'cupy'` / `'numpy'` | GPU vs CPU SFFT backend |
| `template_subtraction.sfft_cuda_device` | `'0'` | CUDA device seen by each worker |
| `template_subtraction.forceconv` | `'REF'` | Convolve the template to the science PSF so the science ePSF applies to the difference image |
| `template_subtraction.kernel_order` | `'0'` | Kernel polynomial degree; `'auto'` adds spatial terms only with supporting evidence |
| `template_subtraction.zogy_psf` | `'auto'` | Reuse the photometry ePSF for ZOGY, else a fitted Moffat |
| `template_subtraction.sfft_memory_policy` | `'fallback'` | Response to an over-budget SFFT estimate: `'shrink'`, `'fallback'`, or `'ignore'` |
| `template_subtraction.sfft_max_memory_gb` | `16` | SFFT peak-memory estimate budget; `null` auto-derives from free RAM |
| `template_subtraction.sfft_num_threads` | `4` | BLAS threads per SFFT run |
| `template_subtraction.match_max_sources` | `2000` | Cap on the matched-source list used for kernel fitting |
| `template_subtraction.download_templates` | `'panstarrs'` | Fetch templates before subtracting |

- `sfft_memory_policy` values: `'shrink'` reduces the kernel to fit, `'fallback'` switches to HOTPANTS with SFFT left intact, `'ignore'` logs the estimate and runs the full kernel
- With `nGPU` worker pinning each subprocess already sees only its assigned GPU, so keep `sfft_cuda_device` at `'0'`

In [ ]:
autophot_input['template_subtraction']['do_subtraction'] = 1
autophot_input['template_subtraction']['alignment_method'] = 'spalipy'
autophot_input['template_subtraction']['method'] = 'sfft'

autophot_input['template_subtraction']['sfft_backend'] = 'cupy' if _HAS_GPU else 'numpy'
autophot_input['template_subtraction']['sfft_cuda_device'] = '0'
autophot_input['template_subtraction']['forceconv'] = 'REF'
autophot_input['template_subtraction']['kernel_order'] = '0'
autophot_input['template_subtraction']['zogy_psf'] = 'auto'
autophot_input['template_subtraction']['sfft_memory_policy'] = 'fallback'

# autophot_input['template_subtraction']['sfft_max_memory_gb'] = 16
# autophot_input['template_subtraction']['sfft_num_threads'] = 4
# autophot_input['template_subtraction']['match_max_sources'] = 2000
# autophot_input['template_subtraction']['download_templates'] = 'panstarrs'

## 7. Parallelism and verbosity

Worker counts and console output.

| Key | Set to | Purpose |
|---|---|---|
| `nCPU` | `1` | Images processed in parallel; `AUTOPHOT_NCPU` overrides |
| `nGPU` | `0` | GPUs the image workers are pinned across round-robin; `0` keeps the ambient device mask |
| `global_verbose_level` | `1` | Console verbosity `0` quiet to `3` debug; per-image `LOG_*.log` files record everything either way |

- Worker settings like `photometry.aperture_n_jobs` multiply `nCPU`, so keep the product at or below your core count

In [ ]:
autophot_input['nCPU'] = 1
autophot_input['nGPU'] = 0
autophot_input['global_verbose_level'] = 1

## 8. Run

Executes the pipeline and returns the path of the collected output CSV. Pass `do_photometry=False` to collect existing outputs without reprocessing.

In [ ]:
loc = AutomatedPhotometry.run_photometry(default_input=autophot_input)
print(loc)

## 9. Light curve and tables

Post-run products from `lightcurve.py`.

| Function | Purpose |
|---|---|
| `plot_lightcurve` | Detections plus upper limits; returns the detections CSV |
| `plot_variability_check` | Target scatter against the reference-star ensemble, separating real variability from instrumental drift |
| `check_detection_plots` | Sorts the per-image detection plots into folders |
| `generate_photometry_table` | ASCII photometry table |

- Light curves for `additional_targets` are written under the `sub_targets/` folder next to the main output

In [ ]:
from lightcurve import (
    plot_lightcurve, plot_variability_check,
    check_detection_plots, generate_photometry_table,
)

_primary_name = autophot_input.get('target_name', 'Main target')

detections_loc = plot_lightcurve(
    loc, snr_limit=3, method='PSF', format='png', show=True,
    target_name=_primary_name,
)

try:
    plot_variability_check(loc, method='PSF', target_name=_primary_name, show=True)
except Exception as e:
    print(f'[WARNING] Variability check failed: {e}')

check_detection_plots(detections_loc, method='AP')
generate_photometry_table(loc, snr_limit=3, method='PSF', target_name=_primary_name)